# Gold Layer – Pharmacy Distribution by Disease

## Overview
This notebook creates an aggregated Gold table that measures pharmacy distribution for each disease.

The dataset is derived from the main analytical Gold table:

`capstone.gold.disease_medication_pharmacy_gold`

and aggregates pharmacy coverage across diseases.

---

## Business Objective
Healthcare analysts and stakeholders often need to understand:

- How widely medications for a specific disease are distributed
- Which diseases have the largest pharmacy coverage
- Which diseases have limited pharmacy availability

This table supports answering the following question:

**"For each disease, how many pharmacies provide medications associated with that disease?"**

---

## Source Table
The table is built from the following Gold dataset:

| Layer | Table |
|------|------|
| Gold | `capstone.gold.disease_medication_pharmacy_gold` |

---

## Target Table

`capstone.gold.pharmacy_count_disease_gold`

---

## Table Grain

One row represents:

**one disease → aggregated pharmacy coverage**

---

## Key Metrics Produced

- `pharmacy_count` – number of unique pharmacies that provide medications linked to the disease.

---

## Downstream Usage

This dataset is used for:

- Pharmacy coverage dashboards
- Disease accessibility analysis
- Healthcare market availability insights

In [0]:
%sql
-- ============================================================
-- Gold Table: pharmacy_count_disease_gold
--
-- Purpose:
-- Create an aggregated Gold dataset that measures pharmacy
-- distribution per disease using the main analytical dataset.
--
-- Business Question:
-- How many pharmacies provide medications associated
-- with each disease?
--
-- Source Table:
-- capstone.gold.disease_medication_pharmacy_gold
--
-- Target Table:
-- capstone.gold.pharmacy_count_disease_gold
-- ============================================================

CREATE OR REPLACE TABLE capstone.gold.pharmacy_count_disease_gold
USING DELTA
AS

SELECT
    disease_id,
    disease_name,

    -- Count distinct pharmacies that provide medications
    COUNT(DISTINCT pharmacy_id) AS pharmacy_count

FROM capstone.gold.disease_medication_pharmacy_gold

GROUP BY
    disease_id,
    disease_name;

In [0]:
%sql
-- ============================================================
-- Data Quality Validation
--
-- These checks verify that:
-- 1. No critical fields are missing
-- 2. No duplicate diseases exist
-- 3. Pharmacy counts are valid
-- ============================================================

SELECT 'Total Rows' AS check_name, COUNT(*) AS result
FROM capstone.gold.pharmacy_count_disease_gold

UNION ALL

SELECT 'Missing Disease ID', COUNT(*)
FROM capstone.gold.pharmacy_count_disease_gold
WHERE disease_id IS NULL OR TRIM(disease_id) = ''

UNION ALL

SELECT 'Missing Disease Name', COUNT(*)
FROM capstone.gold.pharmacy_count_disease_gold
WHERE disease_name IS NULL OR TRIM(disease_name) = ''

UNION ALL

SELECT 'Negative Pharmacy Count', COUNT(*)
FROM capstone.gold.pharmacy_count_disease_gold
WHERE pharmacy_count < 0

UNION ALL

SELECT 'Duplicate Disease Rows', COUNT(*)
FROM (
    SELECT
        disease_id,
        COUNT(*) AS cnt
    FROM capstone.gold.pharmacy_count_disease_gold
    GROUP BY disease_id
    HAVING COUNT(*) > 1
) dup;

In [0]:
%sql
-- Preview the diseases with the largest pharmacy coverage

SELECT
    disease_name,
    pharmacy_count
FROM capstone.gold.pharmacy_count_disease_gold
ORDER BY pharmacy_count DESC
LIMIT 20;

In [0]:
%sql
-- Dataset summary statistics

SELECT
    COUNT(*) AS total_diseases,
    MIN(pharmacy_count) AS min_pharmacy_coverage,
    MAX(pharmacy_count) AS max_pharmacy_coverage,
    ROUND(AVG(pharmacy_count),2) AS avg_pharmacy_coverage
FROM capstone.gold.pharmacy_count_disease_gold;